# Choose reduction axes

This notebook uses Rainbow Tensor 1.2 or newer. Start with a small array, predict which axes remain, then inspect one output. The same axis choices work for `sum` and `mean`.

Rainbow Tensor returns a visual explanation. We use NumPy when we need a numerical result array for the next calculation.


In [ ]:
import numpy as np
from IPython.display import display

import rainbow_tensor as rt

x = np.arange(24).reshape(2, 3, 4)
rt.shape(x)


## Reduce one axis

The source shape is `(2, 3, 4)`. If axis 1 disappears, axes 0 and 2 remain, so the result shape should be `(2, 4)`. Each output sums three source values.


In [ ]:
single = rt.sum(x, axis=1)
assert single.result_shape == np.sum(x, axis=1).shape == (2, 4)
assert single.trace.term_count == 3
single


## Reduce every axis, or no axes

Omitting `axis` is the same as `axis=None`. Every source value contributes to one scalar result. Its shape is `()` and its focus coordinate is also `()`.

An empty axis tuple has a different meaning. `axis=()` reduces no axes, so every output keeps one source value. A mean then divides by 1.


In [ ]:
whole = rt.sum(x)
assert whole.result_shape == np.sum(x, axis=None).shape == ()
assert whole.trace.output_coord == ()
assert whole.trace.term_count == 24
assert x.sum() == 276
whole


In [ ]:
identity = rt.mean(x, axis=(), focus=(1, 1, 2))
assert identity.result_shape == np.mean(x, axis=()).shape == x.shape
assert identity.trace.term_count == 1
assert identity.trace.divisor == 1
assert identity.trace.terms[0][0].coordinate == (1, 1, 2)
identity


## Average two axes together

Choose axes `(0, 2)`. Only axis 1 remains, so the output has shape `(3,)`. Each output uses `2 * 4 = 8` values. Focus on output `(1,)`: its source values are `4, 5, 6, 7, 16, 17, 18, 19`, whose mean is `11.5`.


In [ ]:
multi = rt.mean(x, axis=(0, 2), focus=(1,))
assert multi.result_shape == np.mean(x, axis=(0, 2)).shape == (3,)
assert multi.trace.term_count == multi.trace.divisor == 8
coordinates = [term[0].coordinate for term in multi.trace.terms]
values = [int(x[coordinate]) for coordinate in coordinates]
assert values == [4, 5, 6, 7, 16, 17, 18, 19]
assert sum(values) / multi.trace.divisor == np.mean(x, axis=(0, 2))[1] == 11.5
multi


## Tuple order and negative axes

`(2, 0)` chooses the same axes as `(0, 2)`. For this three-dimensional source, `(-3, -1)` also means `(0, 2)`.

Reduction traces follow source row-major coordinate order, with the last reduced source axis changing fastest. Changing the order of the axis tuple does not change the contribution order. An axis must not appear twice in a tuple.


In [ ]:
reordered = rt.mean(x, axis=(2, 0), focus=(1,))
negative = rt.mean(x, axis=(-3, -1), focus=(1,))
assert reordered.trace.terms == negative.trace.terms == multi.trace.terms
assert negative.metadata["reduction"] == {
    "axes": (0, 2), "keepdims": False, "term_count": 8,
}
negative


## Keep reduced axes at length one

`keepdims=True` is a keyword option. It changes the result shape from `(3,)` to `(1, 3, 1)`, without changing which eight values contribute.

The same focused group now has coordinate `(0, 1, 0)`. The retained reduced axes use the accent colour. The surviving axis keeps its original colour.


In [ ]:
kept = rt.mean(x, axis=(0, 2), keepdims=True, focus=(0, 1, 0))
assert kept.result_shape == np.mean(x, axis=(0, 2), keepdims=True).shape == (1, 3, 1)
assert kept.trace.terms == multi.trace.terms
assert kept.metadata["reduction"]["keepdims"] is True
kept


## Why keepdims helps normalize rows

Each row below contains three scores. Divide a row by its own total to turn the scores into fractions that sum to one.

The totals are 12 and 18. Keeping the reduced axis gives shape `(2, 1)`, so each total can stretch across its row. Without `keepdims`, shape `(2,)` cannot align with the 3 columns when broadcasting compares axes from the right.


In [ ]:
scores = np.array([[2., 4., 6.], [3., 6., 9.]])
row_visual = rt.sum(scores, axis=-1, keepdims=True, focus=(1, 0))
assert row_visual.result_shape == (2, 1)
row_visual


In [ ]:
row_totals = np.sum(scores, axis=1, keepdims=True)
np.testing.assert_array_equal(row_totals, [[12.], [18.]])
rt.broadcast(scores, row_totals)


In [ ]:
normalized = scores / row_totals
np.testing.assert_allclose(normalized.sum(axis=1), [1., 1.])
np.testing.assert_allclose(normalized, [[1 / 6, 1 / 3, 1 / 2]] * 2)
rt.shape(normalized)


## Keep large examples bounded

`max_terms` limits contributions per output. For multiple reduced axes, that count is the product of their sizes. `max_total_terms` limits contributions across the planned visible output panel. If either limit is exceeded, the outputs show `?`, without a partial answer.

A trace keeps at most eight terms, but still reports the full term count and the full mean divisor. The shape tuples below generate placeholder values without allocating these arrays. Displayed source cells are still read or generated.


In [ ]:
budgeted = rt.mean((2, 3, 10_000), axis=(0, 2), keepdims=True)
assert budgeted.metadata["value_evaluation"]["status"] == "skipped"
assert budgeted.metadata["value_evaluation"]["reason"] == "max_terms"
assert budgeted.trace.term_count == budgeted.trace.divisor == 20_000
assert len(budgeted.trace.terms) == 8
assert not budgeted.trace.complete
budgeted


In [ ]:
total_limited = rt.sum((20, 10_000), axis=-1)
assert total_limited.metadata["value_evaluation"]["reason"] == "max_total_terms"
total_limited


## Optional: move the focus

If `rainbow-tensor[interactive]` is installed, the next cell opens notebook controls. In result shape `(1, 3, 1)`, change the middle coordinate and press **Update focus**. The size-one coordinates stay at 0. Both calculation budgets remain in effect.

The static examples above do not need widgets. Live controls need a running notebook kernel and widget support in its host.


In [ ]:
from importlib.util import find_spec

explorer = None
if find_spec("ipywidgets") is None:
    print('Optional controls require: pip install "rainbow-tensor[interactive]"')
else:
    explorer = rt.explore(rt.mean, x, axis=(0, 2), keepdims=True, focus=(0, 1, 0))
    display(explorer)


Run the next cell after trying the controls. It checks a second focus and closes the widget communications. The last static result remains available as `explorer.visual`.


In [ ]:
if explorer is not None:
    explorer.set_focus((0, 2, 0))
    assert explorer.visual.trace.divisor == 8
    assert explorer.visual.trace.output_coord == (0, 2, 0)
    explorer.close()


## Inputs and numerical values

Scalar sources use shape `()` and empty sources have at least one zero-length axis. Both are supported. `axis=()` means no reduced axes, not an empty input. Continue with `11_scalars_and_empty_tensors.ipynb` to compare empty groups with empty results.

The previews use Python scalar arithmetic. Backend rounding, overflow and accumulation dtype may differ. The small integer examples here agree with NumPy, but use NumPy or your tensor framework when you need its native numerical result. Reducing no axes preserves each value without promising an unchanged backend dtype.

Pass `max_terms=None` or `max_total_terms=None` to remove that particular limit. Both must be `None` to remove both arithmetic limits. Changing limits does not expand the eight-term trace sample.
